<a href="https://colab.research.google.com/github/Octavio1200/Finanzas-Computacionales/blob/main/Copia_de_MonteCEuro4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Valuación opciones europeas por Simulación Monte Carlo**


- La simulación de Monte Carlo comprende una clase de algoritmos computacionales que utilizan muestreo
  aleatorio con repeticiones para resolver un problema que admita una interpretación probabilística. En finanzas cuantitativas tiene una aplicación práctica ya que se pueden estimar con precisión esperanzas que involucran el cálculo de integrales.


- La idea principal de la simulación Monte Carlo es generar una cantidad suficiente de trayectorias de un
  proceso estocástico en particular, lo que representan los posibles escenarios o resultados durante un  período de tiempo
  determinado. Una vez definida de manera teórica la ecuación diferencial estocástica, el horizonte de tiempo se divide en un número dado de pasos de tiempo, esto se conoce como discretización. Su objetivo es modelar en tiempo continuo tal proceso, ya que el precio de los instrumentos financieros por simplicidad se denota en tiempo continuo. No obstante en la práctica dependen de factores externos que provienen de  la interacción de los agentes que participan en los mercados financieros.
  

- Los resultados de las trayectorias simuladas se pueden usar para obtener algunos cálculos como la frecuencia de las veces que ocurrió un evento, el valor promedio de los precios de un activo en el último paso, etc. Históricamente, el principal problema con el enfoque de Monte Carlo era que se requería una gran potencia computacional para calcular todos los escenarios posibles. Actualmente este es un problema menor,  ya que se pueden ejecutar simulaciones en cualquier computadora o en la nube, como google colab.

    
- Monte Carlo es una de las técnicas más importantes en finanzas computacionales. Se puede adaptar a diversos  problemas, como la valoración de derivados cuya solución de forma cerrada no es analítica, v.g. opciones exóticas, valuación de bonos, por ejemplo, bono cupón cero, estimar métricas de riesgo de un portafolio: cómo Valor en Riesgo  y déficit esperado, o la realización de pruebas de stress en administración de riesgos.    

## **1. Cargamos librerías**

In [ ]:
###

## **2. Fórmula de Black-Scholes-Merton**
### **2.1 Recordamos fórmulas**

El precio de una opción de compra europea sobre un activo subyacente $S_t$ sin pago de dividendos al tiempo $t$ con precio de ejercicio $K$ y
vencimiento en $T$, $c=c(S_t,t;T,K,r,\sigma)$ está dado por:
    
  
$$
\begin{equation}
c = {S_t}\Phi ({d_1}) - {e^{ - r(T - t)}}K\Phi ({d_2})
\end{equation}
$$
y el de una opción de venta europea por:
    
$$
p=Ke^{-r(T-t)} \Phi(-d_2)-S_t \Phi(-d_1).
$$
   
donde la función $\Phi(d)$ es la función de distribución
acumulada de ${\cal E}\sim{\cal N}(0,1),$ es decir,

$$
    \Phi(d) ={P}_{\cal E}\{{\cal E}\le d\}
            =\int_{-\infty}^{d}{1\over\sqrt{2\pi}}e^{-{1\over2}\epsilon^2}d\epsilon
           =1-\Phi(-d),
$$
donde:
$$
d_1=d_1(S_t,t;T,K,r,\sigma)={{\ln\left({{\hbox{$S_t$}\over\hbox{$
K$}}}\right) +(r+{1\over 2}\sigma^{2})(T-t)}\over
{\sigma\sqrt{T-t}}}
$$
y
$$
{d_2} = {d_2}({S_t},t;T,K,r,\sigma ) = \frac{{\ln \left( {\frac{{{S_t}}}{K}} \right) +
(r - \frac{1}{2}{\sigma ^2})(T - t)}}{{\sigma \sqrt {T - t} }}
 = {d_1} - \sigma \sqrt {T - t}.
$$


### **2.2 Función que aplica fórmula analítica del modelo BSM**

In [ ]:
def black_scholes_formula(S_0, K, T, r, sigma, type='call'):
    '''
   Función utilizada para calcular el precio opción europea con la
   fórmula analítica del modelo de Black-Scholes.

    Parámetros de entrada
    ------------
    s_0 : float
          Precio Inicial del activo
    K : float
        Precio de ejercicio
    T : float
        Plazo al vencimiento en años
    r : float
       Tasa libre de riesgo anualizada
    sigma : float
        Desviación estándar de los rendimientos del activo
    type : str
        Tipo de opción. Compra: call. Venta: put.
        Admitidos: ['call', 'put']

    Salidas
    -----------
    prima_opcion : float
     La prima de la opción calculada con el modelo de Black-Scholes.
    '''

    d1 = (np.log(S_0 / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = (np.log(S_0 / K) + (r - 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))

    if type == 'call':
        prima_opcion = (S_0 * norm.cdf(d1, 0, 1) - K * np.exp(-r * T) * norm.cdf(d2, 0, 1))
    elif type == 'put':
        prima_opcion = (K * np.exp(-r * T) * norm.cdf(-d2, 0, 1) - S_0 * norm.cdf(-d1, 0, 1))
    else:
        raise ValueError('Entrada incorrecta para el tipo de opción. Corregir.')

    return prima_opcion

## **3. Definimos parámetros de la opción**

In [ ]:
###

In [ ]:
type(dt)

In [ ]:
# Calcula call
bs_call_cerrada=black_scholes_formula(S_0=S_0, K=K, T=T, r=r, sigma=sigma, type='call')
print(f'Precio de la opción de compra: ${bs_call_cerrada:.8f}')

In [ ]:
###
# Calcula put

### **3.1 Función que simula trayectorias del MGB**

In [ ]:
def simular_mgb(s_0, mu, sigma, n_sims, T, N):
    '''
    Función utilizada para simular precios de un activo con Mov. Geométrico Browniano.

    Parametros
    ------------
    s_0 : float
          Precio Inicial del activo
    mu : float
         Coeficiente de tendencia (Drift)
    sigma : float
         Coeficiente de volatilidad
    n_sims : int
        Número de trayectorias de la simulación
    dt : float
        Incremento de tiempo, usualmente un día
    T : float
        Longitud del horizonte de pronóstico, misma unidad que dt
    N : int
        Número de incrementos de tiempo en el horizonte de pronóstico
    #random_seed : int
        Semilla aleatoria para reproducir

    Salidas
    -----------
    S_t : np.ndarray
        Matriz (tamaño: n_sims x (T + 1)) que contiene los resultados de la simulación.
        Las filas representan trayectorias, mientras que las columnas representan el tiempo.
    '''
    np.random.seed(1234) # Semilla, en teoría los resultados que obtengamos son iguales.

    dt = T/N
    dW = np.random.normal(scale = np.sqrt(dt), size=(n_sims, N))
    W = np.cumsum(dW, axis=1)

    tiempo_paso = np.linspace(dt, T, N)
    tiempo_pasos = np.broadcast_to(tiempo_paso, (n_sims, N))

    S_t = s_0 * np.exp((mu - 0.5 * sigma**2) * tiempo_pasos
                       + sigma * W)
    S_t = np.insert(S_t, 0, s_0, axis=1)

    return S_t

In [ ]:
###
# Generamos trayectorias con la función anterior


In [ ]:
###

In [ ]:
###


In [ ]:
###


In [ ]:
###


In [ ]:
###


In [ ]:
###


In [ ]:
new_mgb_sims.shape

In [ ]:
# Graficamos 100 trayectorias
plt.figure(figsize=(10, 7))
plt.plot(new_mgb_sims[:,:100]);
#La ruta se puede cambiar
plt.savefig('MGB1.png')
plt.savefig('MGB1.pdf')

In [ ]:
# Graficamos 1000 trayectorias
#tiempo_paso = np.linspace(dt, T, N)
plt.figure(figsize=(10, 7))
plt.plot(np.transpose(mgb_sims[:100,:]))
plt.show()

In [ ]:
#Valores finales de mgb_sims
mgb_sims[:, -1]


In [ ]:
mgb_sims[:, -1].shape

In [ ]:
# Verificamos el supuesto de precios lognormales
from scipy.stats import lognorm
valsFinS=mgb_sims[:, -1]
param_LN = lognorm.fit(valsFinS)
#
x = np.linspace(valsFinS.min(), valsFinS.max(), 100)
pdf_LN_fitted = lognorm.pdf(x, *param_LN)
#print(pdf_LN_fitted)
#
plt.figure(figsize=(10, 7))
plt.plot(x, pdf_LN_fitted, color='r', lw=3, label="Log-Normal")
plt.hist(valsFinS, density=True, bins=100, facecolor="green", edgecolor='w',label="Frecuencia de ValsFinS")
plt.legend(fontsize=18);
plt.title("Histograma vs Log-Normal");
plt.xlabel("S_T")
plt.show()

In [ ]:
# Verificamos el supuesto de precios lognormales
from scipy.stats import lognorm
valsFinST=np.transpose(mgb_sims[:, -1])
param_LN = lognorm.fit(valsFinST)
#
x = np.linspace(valsFinST.min(), valsFinST.max(), 100)
pdf_LN_fitted = lognorm.pdf(x, *param_LN)
#print(pdf_LN_fitted)
#
plt.figure(figsize=(10, 7))
plt.plot(x, pdf_LN_fitted, color='violet', lw=3, label="Log-Normal")
plt.hist(valsFinST, density=1, bins=100, facecolor="hotpink", edgecolor='w',label="Frecuencia de ValsFinS")
plt.legend(fontsize=18);
plt.title("Histograma vs Log-Normal");
plt.xlabel("S_T")
plt.show()

In [ ]:
# Imprime parámetros estimados
print(param_LN)

In [ ]:
#lognorm.pdf(x, s, loc, scale)
# Imprime el valor de cada punto evaluado en la función de densidad de la lognormal
# con los parametros estimados por máxima verosimilitud
print(pdf_LN_fitted);

## **4. Valuación teórica con Black-Scholes-Merton, por Monte Carlo e intervalo de confianza de call y put**

In [ ]:
bs_call_cerrada=black_scholes_formula(S_0=S_0, K=K, T=T, r=r, sigma=sigma, type='call')
print(f'Precio de la opción de compra: ${bs_call_cerrada:.6f}')

In [ ]:
# Precio de opción de compra por Monte Carlo ###
prima_call_MC = factor_descuento * np.average(np.maximum(0, mgb_sims[:, -1] - K))
print(f'Precio de la opción de compra Monte Carlo: ${prima_call_MC:.6f}')

#### **4.1 Intervalo de confianza para opción de compra**

In [ ]:
###

### **Para la opción de venta: Put**

In [ ]:
bs_put_cerrada=black_scholes_formula(S_0=S_0, K=K, T=T, r=r, sigma=sigma, type='put')
print(f'Precio de la opción de venta: ${bs_put_cerrada:.6f}')

In [ ]:
# Precio de opción de venta por Monte Carlo ###
prima_put_MC = factor_descuento * np.average(np.maximum(0, K - mgb_sims[:, -1]))
print(f'Precio de la opción de venta Monte Carlo: ${prima_put_MC:.6f}')

#### **4.1 Intervalo de confianza para opción de venta**

In [ ]:
P= factor_descuento*np.maximum(K- mgb_sims[:,-1], 0)
S=np.sqrt(np.sum(pow((P-np.average(np.maximum(K - mgb_sims[:, -1], 0))),2))/(N_SIMS-1))
intervalo_put_inf= prima_put_MC- norm.ppf(0.95)*(S/np.sqrt(N_SIMS))
intervalo_put_sup= prima_put_MC+ norm.ppf(0.95)*(S/np.sqrt(N_SIMS))
print(f'Intervalo de la opcion Europea de venta put: {intervalo_put_inf:.6f},{intervalo_put_sup:.6f}')

## **5. Valuación opciones asiáticas: average price (Precio promedio)**

$$
\begin{array}{l}
{{\hat c}_{AP}} = {e^{ - rT}}\frac{1}{N}\sum\limits_{i = 1}^N {{\rm{max}}\left( {\bar S_T^i - K,0} \right)} \\
{{\hat p}_{AP}} = {e^{ - rT}}\frac{1}{N}\sum\limits_{i = 1}^N {{\rm{max}}\left( {K - \bar S_T^i,0} \right)} \\
\bar S_T^i = \frac{1}{N}\sum\limits_{i = 1}^N {S_T^i}
\end{array}
$$

In [ ]:
prima_call_MC_AveragePrice = factor_descuento * np.average(np.maximum(0, np.average(mgb_sims[:, -1]) - K))
print(f'Precio de la opción de compra AveragePrice por Monte Carlo: ${prima_call_MC_AveragePrice:.6f}')

In [ ]:
prima_put_MC_AveragePrice = factor_descuento * np.average(np.maximum(0,  K - np.average(mgb_sims[:, -1]))) ###
print(f'Precio de la opción de venta AveragePrice por Monte Carlo: ${prima_put_MC_AveragePrice:.6f}')

In [ ]:
# En este caso el promedio resultó mayor que S y K.
# Para las call tenemos precio menor que por fórmula cerrada.
# Para put no, por el resultado del promedio de los valores finales.
promedio_S=np.average(mgb_sims[:, -1]);
promedio_S

In [ ]:
np.maximum(K-promedio_S,0)

In [ ]:
10**6

In [ ]:
#reset

In [ ]:
1/52